## Competitor Sales Analysis

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

import math
import random
np.random.seed(seed=10000)
import time

#pd.set_option("display.max_rows", None) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option("display.max_rows", 1000) # show all rows with scrollbar, don't use if there are many rows.
pd.set_option("display.max_columns", None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
%autosave 180

Autosaving every 180 seconds


In [2]:
revenue_total = pd.read_excel('RevenueTotal.xlsx', header=None).values.flatten()

In [3]:
N = revenue_total.shape[0]

In [4]:
X0, Z0 = pd.read_excel('Initial.xlsx', header=None).values[[0,2]]
Y0 = revenue_total - X0 -Z0

In [5]:
growth_rate = pd.read_excel('GrowthRate.xlsx', header=None)

In [6]:
growth_rate.fillna('Unknown', inplace=True)

In [7]:
growth_rate.at[1,8] = 'Unknown1'
growth_rate.at[1,9] = 'Unknown2'
growth_rate

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,-MSD,+HSD,+HSD,+HSD,+LSD,+LSD,+HSD,+MSD,+HSD,Unknown,+MSD,Unknown,Unknown
1,Unknown,Unknown,Unknown,Unknown,Unknown,-MT,Unknown,+MT,Unknown1,Unknown2,+HSD,+MSD,-LDD
2,-MSD,Unknown,Unknown,Unknown,Unknown,-MSD,Unknown,+MSD,+LDD,+LDD,Unknown,+HSD,Unknown


In [8]:
gr_known_index = growth_rate!='Unknown'
gr_known_index = gr_known_index.values.flatten()

In [9]:
gr_known_index

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
       False,  True, False, False, False, False, False, False, False,
        True, False,  True,  True,  True,  True,  True,  True,  True,
       False, False, False, False,  True, False,  True,  True,  True,
       False,  True, False])

In [10]:
#gr_known_index[:N-4] = False
#gr_known_index[-(N-4):] = False

In [11]:
gr_low_dict = {
 '-HTH':-40, '-MTH':-30-20./3, '-LTH':-30-10./3,
 '-HTM':-30, '-MTW':-20-20./3, '-LTW':-20-10./3,
 '-HT':-20, '-MT':-10-20./3, '-LDD':-10-10./3,
 '-HSD':-10, '-MSD':0-20./3, '-LSD':0-10./3,
 'Flat':-0.01,
 '+LSD':0.01, '+MSD':0+10./3, '+HSD':0+20./3,
 '+LDD':10, '+MT':10+10./3, '+HT':10+20./3,
 '+LTW':20, '+MTW':20+10./3, '+HTW':20+20./3,
 '+LTH':30, '+MTH':30+10./3, '+HTH':30+20./3,
 'Unknown':-40, 'Unknown1':20, 'Unknown2':30}

gr_high_dict = {
 '-HTH':-40+10./3, '-MTH':-30-20./3+10./3, '-LTH':-30,
 '-HTM':-30+10./3, '-MTW':-20-20./3+10./3, '-LTW':-20,
 '-HT':-20+10./3, '-MT':-10-20./3+10./3, '-LDD':-10,
 '-HSD':-10+10./3, '-MSD':0-20./3+10./3, '-LSD':-0.01,
 'Flat':0.01,
 '+LSD':10./3, '+MSD':0+10./3+10./3, '+HSD':10,
 '+LDD':10+10./3, '+MT':10+10./3+10./3, '+HT':20,
 '+LTW':20+10./3, '+MTW':20+10./3+10./3, '+HTW':30,
 '+LTH':30+10./3, '+MTH':30+10./3+10./3, '+HTH':40,
 'Unknown':+40, 'Unknown1':30, 'Unknown2':40}

In [12]:
gr_low = (growth_rate.replace(gr_low_dict)/100).values.flatten()
gr_high = (growth_rate.replace(gr_high_dict)/100).values.flatten()

In [13]:
gr_low
gr_high

array([-6.66666667e-02,  6.66666667e-02,  6.66666667e-02,  6.66666667e-02,
        1.00000000e-04,  1.00000000e-04,  6.66666667e-02,  3.33333333e-02,
        6.66666667e-02, -4.00000000e-01,  3.33333333e-02, -4.00000000e-01,
       -4.00000000e-01, -4.00000000e-01, -4.00000000e-01, -4.00000000e-01,
       -4.00000000e-01, -4.00000000e-01, -1.66666667e-01, -4.00000000e-01,
        1.33333333e-01,  2.00000000e-01,  3.00000000e-01,  6.66666667e-02,
        3.33333333e-02, -1.33333333e-01, -6.66666667e-02, -4.00000000e-01,
       -4.00000000e-01, -4.00000000e-01, -4.00000000e-01, -6.66666667e-02,
       -4.00000000e-01,  3.33333333e-02,  1.00000000e-01,  1.00000000e-01,
       -4.00000000e-01,  6.66666667e-02, -4.00000000e-01])

array([-0.03333333,  0.1       ,  0.1       ,  0.1       ,  0.03333333,
        0.03333333,  0.1       ,  0.06666667,  0.1       ,  0.4       ,
        0.06666667,  0.4       ,  0.4       ,  0.4       ,  0.4       ,
        0.4       ,  0.4       ,  0.4       , -0.13333333,  0.4       ,
        0.16666667,  0.3       ,  0.4       ,  0.1       ,  0.06666667,
       -0.1       , -0.03333333,  0.4       ,  0.4       ,  0.4       ,
        0.4       , -0.03333333,  0.4       ,  0.06666667,  0.13333333,
        0.13333333,  0.4       ,  0.1       ,  0.4       ])

In [16]:
t = revenue_total
x = X0[1:]
y = Y0[1:]
z = Z0[1:]
sigma2_gr = 0.001
sigma2_bi = 1
sigma_d = 20
sigma2_dx = (sigma_d*300/revenue_total[:4].sum())**2
sigma2_dy = (sigma_d*(revenue_total[:4].sum()-300-250)/revenue_total[:4].sum())**2
sigma2_dz = (sigma_d*250/revenue_total[:4].sum())**2

sigma = 2
sigma_x = sigma*300/revenue_total[:4].sum()
sigma_y = sigma*(revenue_total[:4].sum()-300-250)/revenue_total[:4].sum()
sigma_z = sigma*250/revenue_total[:4].sum()

X_all = []
Y_all = []
Z_all = []
X_all_best = []
Y_all_best = []
Z_all_best = []
N_iter = 100000
N_solutions = 8
x_max = 108
x_min = 60
z_max = 80
z_min = 50

In [17]:
i = 0
while True:
    i = i + 1
    x_ = x + np.random.normal(0.0, sigma_x*20, len(x))
    z_ = z + np.random.normal(0.0, sigma_z*20, len(z))
    x1_ = 300-x_[:3].sum()
    z1_ = 250-z_[:3].sum()
    X_ = np.concatenate(([x1_], x_))
    Z_ = np.concatenate(([z1_], z_))
    # boundary conditions
    if x1_<=0 or z1_<=0:
        continue
    if any(X_>x_max) or any(X_<x_min):
        continue
    if any(Z_>z_max) or any(Z_<z_min):
        continue
    x = x_
    z = z_
    print(i)
    break

for i in range(N_iter):
#     if i%1000 == 1:
#         print('iter = ', i)
    
    x1 = 300-x[:3].sum()
    z1 = 250-z[:3].sum()
    X = np.concatenate(([x1], x))
    Z = np.concatenate(([z1], z))
    Y = t - X - Z
    gr_X = (X[4:] - X[:-4])/X[:-4]
    gr_Z = (Z[4:] - Z[:-4])/Z[:-4]
    gr_Y = (Y[4:] - Y[:-4])/Y[:-4]
    gr = np.concatenate((gr_X, gr_Y, gr_Z))
    
    #logp = -0.5*(np.diff(X)**2/sigma2_dx + np.diff(t-X-Z)**2/sigma2_dy + np.diff(Z)**2/sigma2_dz).sum()
    #logp = -0.5*((gr-gr_high)**2+(gr-gr_low)**2).sum()/sigma2_gr
    #logp = -0.5*((gr-gr_high)**2+(gr-gr_low)**2)[gr_known_index].sum()/sigma2_gr
    #logp = -(sum(gr>gr_high)+sum(gr<gr_low))/sigma2_bi
    #logp = -(sum(gr[gr_known_index]>gr_high[gr_known_index])+sum(gr[gr_known_index]<gr_low[gr_known_index]))/sigma2_bi
    #logp = -(sum(gr>gr_high)+sum(gr<gr_low))/sigma2_bi*0.5*((gr-gr_high)**2+(gr-gr_low)**2).sum()/sigma2_gr
    #logp = -(sum(gr>gr_high)+sum(gr<gr_low))/sigma2_bi*0.5*((gr-gr_high)**2+(gr-gr_low)**2)[gr_known_index].sum()/sigma2_gr
    logp = -(sum(gr>gr_high)+sum(gr<gr_low))/sigma2_bi*0.5*(((gr-gr_high)**2+(gr-gr_low)**2)[gr_known_index].sum()+0.01*((gr-gr_high)**2+(gr-gr_low)**2)[~gr_known_index].sum())/sigma2_gr

    x_ = x + np.random.normal(0.0, sigma_x, len(x))
    z_ = z + np.random.normal(0.0, sigma_z, len(z))
    x1_ = 300-x_[:3].sum()
    z1_ = 250-z_[:3].sum()
    X_ = np.concatenate(([x1_], x_))
    Z_ = np.concatenate(([z1_], z_))
    Y_ = t - X_ - Z_
    # boundary conditions
    if x1_<=0 or z1_<=0:
        continue
    if any(X_>x_max) or any(X_<x_min):
        continue
    if any(Z_>z_max) or any(Z_<z_min):
        continue
    gr_X_ = (X_[4:] - X_[:-4])/X_[:-4]
    gr_Z_ = (Z_[4:] - Z_[:-4])/Z_[:-4]
    gr_Y_ = (Y_[4:] - Y_[:-4])/Y_[:-4]
    gr_ = np.concatenate((gr_X_, gr_Y_, gr_Z_))
    
    #logp_ = -0.5*(np.diff(X_)**2/sigma2_dx + np.diff(t-X_-Z_)**2/sigma2_dy + np.diff(Z_)**2/sigma2_dz).sum()
    #logp_ = -0.5*((gr_-gr_high)**2+(gr_-gr_low)**2).sum()/sigma2_gr
    #logp_ = -0.5*((gr_-gr_high)**2+(gr_-gr_low)**2)[gr_known_index].sum()/sigma2_gr
    #logp_ = -(sum(gr_>gr_high)+sum(gr_<gr_low))/sigma2_bi
    #logp_ = -(sum(gr_[gr_known_index]>gr_high[gr_known_index])+sum(gr_[gr_known_index]<gr_low[gr_known_index]))/sigma2_bi
    #logp_ = -(sum(gr_>gr_high)+sum(gr_<gr_low))/sigma2_bi*0.5*((gr_-gr_high)**2+(gr_-gr_low)**2).sum()/sigma2_gr
    #logp_ = -(sum(gr_>gr_high)+sum(gr_<gr_low))/sigma2_bi*0.5*((gr_-gr_high)**2+(gr_-gr_low)**2)[gr_known_index].sum()/sigma2_gr
    logp_ = -(sum(gr_>gr_high)+sum(gr_<gr_low))/sigma2_bi*0.5*(((gr_-gr_high)**2+(gr_-gr_low)**2)[gr_known_index].sum()+0.01*((gr_-gr_high)**2+(gr_-gr_low)**2)[~gr_known_index].sum())/sigma2_gr

    #alpha = np.exp(logp_-logp) #*(sum(gr>gr_high)+sum(gr<gr_low))/(sum(gr_>gr_high)+sum(gr_<gr_low))
    alpha = logp_-logp
    #if alpha > random.random():
    if alpha >= 0:
        x = x_
        z = z_
        loss = sum(gr_>gr_high)+sum(gr_<gr_low)
        #loss = sum(gr_[gr_known_index]>gr_high[gr_known_index])+sum(gr_[gr_known_index]<gr_low[gr_known_index])
        print('Binomial Loss =', loss, ', alpha =', alpha)
        #if all(gr_ <= gr_high) and all(gr_ >= gr_low):
        if loss <= 1:
            X_all.append(X_)
            Y_all.append(Y_)
            Z_all.append(Z_)
        if loss <= 1 and all(gr_[gr_known_index] <= gr_high[gr_known_index]) and all(gr_[gr_known_index] >= gr_low[gr_known_index]):
            X_all_best.append(X_)
            Y_all_best.append(Y_)
            Z_all_best.append(Z_)
            print('iter =', i, '; Number of Solutions =', len(X_all_best))
    #print('L2 Loss = ', ((gr_-gr_high)**2+(gr_-gr_low)**2).sum())   
#     if len(X_all) == N_solutions:
#         break

33
Binomial Loss = 22 , alpha = 698.6368906370299
Binomial Loss = 22 , alpha = 34.34466370395239
Binomial Loss = 22 , alpha = 223.40353295745444
Binomial Loss = 20 , alpha = 559.4889436020239
Binomial Loss = 19 , alpha = 538.5846661491341
Binomial Loss = 19 , alpha = 158.12637514703601
Binomial Loss = 20 , alpha = 210.18228917213855
Binomial Loss = 19 , alpha = 669.7102885434924
Binomial Loss = 20 , alpha = 258.2100823968758
Binomial Loss = 20 , alpha = 207.65458693839082
Binomial Loss = 20 , alpha = 197.75222203711837
Binomial Loss = 20 , alpha = 395.2514991803728
Binomial Loss = 20 , alpha = 75.46645130321212
Binomial Loss = 20 , alpha = 10.895531402257802
Binomial Loss = 19 , alpha = 346.8637782667138
Binomial Loss = 18 , alpha = 488.4161341279205
Binomial Loss = 18 , alpha = 89.80793617105974
Binomial Loss = 19 , alpha = 334.714374819554
Binomial Loss = 19 , alpha = 311.83494677833096
Binomial Loss = 20 , alpha = 78.41125597218888
Binomial Loss = 20 , alpha = 394.7256393118705
Bino